In [12]:
from pystac_client import Client

# Connecting to Sentinel-2
# Copernicus Data Space STAC catalog
STAC_URL = "https://stac.dataspace.copernicus.eu/v1"

catalog = Client.open(STAC_URL)

print("Connected to Copernicus!")

Connected to Copernicus!


In [13]:
# coordinates of a plantation to look at
plantation = {
    "type": "Polygon",
    "coordinates": [[
        [longitude_west, latitude_south],
        [longitude_east, latitude_south],
        [longitude_east, latitude_north],
        [longitude_west, latitude_north],
        [longitude_west, latitude_south]
    ]]
}

NameError: name 'longitude_west' is not defined

In [ ]:
plantation = {
    "type": "Polygon",
    "coordinates": [[
        [100.123, 3.123],
        [100.153, 3.123],
        [100.153, 3.153],
        [100.123, 3.153],
        [100.123, 3.123]
    ]]
}

In [ ]:
# Search Sentinel-2 for planation
search = catalog.search(
    collections=["sentinel-2-l2a"],
    intersects=plantation,
    datetime="2026-01-01/2026-09-09",
    query={
        "eo:cloud_cover": {
            "lte": 20
        }
    },
    max_items=100
)

items = list(search.items())

print(f"Found {len(items)} Sentinel-2 images")

Found 1 Sentinel-2 images


In [ ]:
# Look at details of image
item = items[0]

print("ID:", item.id)
print("Date:", item.datetime)
print("Cloud cover:", item.properties.get("eo:cloud_cover"))

ID: S2C_MSIL2A_20260527T032511_N0512_R018_T47NPD_20260527T082311
Date: 2026-05-27 03:25:11.025000+00:00
Cloud cover: 14.17


In [ ]:
# Look at all available images
for item in items:
    print(
        item.datetime.date(),
        round(item.properties.get("eo:cloud_cover", 999), 2),
        item.id
    )

2026-05-27 14.17 S2C_MSIL2A_20260527T032511_N0512_R018_T47NPD_20260527T082311


In [ ]:
# %%
item = items[0]

print("Image ID:")
print(item.id)

print("\nDate:")
print(item.datetime)

print("\nAvailable assets:")
for key, asset in item.assets.items():
    print(key, "->", asset.title)

Image ID:
S2C_MSIL2A_20260527T032511_N0512_R018_T47NPD_20260527T082311

Date:
2026-05-27 03:25:11.025000+00:00

Available assets:
AOT_10m -> Aerosol optical thickness (AOT) - 10m
AOT_20m -> Aerosol optical thickness (AOT) - 20m
AOT_60m -> Aerosol optical thickness (AOT) - 60m
B01_20m -> Coastal aerosol (band 1) - 20m
B01_60m -> Coastal aerosol (band 1) - 60m
B02_10m -> Blue (band 2) - 10m
B02_20m -> Blue (band 2) - 20m
B02_60m -> Blue (band 2) - 60m
B03_10m -> Green (band 3) - 10m
B03_20m -> Green (band 3) - 20m
B03_60m -> Green (band 3) - 60m
B04_10m -> Red (band 4) - 10m
B04_20m -> Red (band 4) - 20m
B04_60m -> Red (band 4) - 60m
B05_20m -> Red edge 1 (band 5) - 20m
B05_60m -> Red edge 1 (band 5) - 60m
B06_20m -> Red edge 2 (band 6) - 20m
B06_60m -> Red edge 2 (band 6) - 60m
B07_20m -> Red edge 3 (band 7) - 20m
B07_60m -> Red edge 3 (band 7) - 60m
B08_10m -> NIR 1 (band 8) - 10m
B09_60m -> NIR 3 (band 9) - 60m
B11_20m -> SWIR 1 (band 11) - 20m
B11_60m -> SWIR 1 (band 11) - 60m
B12_20

In [ ]:
# %%
# Configure Copernicus S3 credentials

import os
from dotenv import load_dotenv

load_dotenv()

os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("CDSE_S3_ACCESS_KEY")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("CDSE_S3_SECRET_KEY")

os.environ["AWS_S3_ENDPOINT"] = "eodata.dataspace.copernicus.eu"
os.environ["AWS_HTTPS"] = "YES"
os.environ["AWS_VIRTUAL_HOSTING"] = "FALSE"

print("Copernicus S3 credentials loaded.")

In [ ]:
# %%
# Get Red and NIR bands
import rasterio

red_asset = item.assets["B04_10m"]
nir_asset = item.assets["B08_10m"]

print("Red URL:")
print(red_asset.href)

print("\nNIR URL:")
print(nir_asset.href)

Red URL:
s3://eodata/Sentinel-2/MSI/L2A/2026/05/27/S2C_MSIL2A_20260527T032511_N0512_R018_T47NPD_20260527T082311.SAFE/GRANULE/L2A_T47NPD_A008997_20260527T033930/IMG_DATA/R10m/T47NPD_20260527T032511_B04_10m.jp2

NIR URL:
s3://eodata/Sentinel-2/MSI/L2A/2026/05/27/S2C_MSIL2A_20260527T032511_N0512_R018_T47NPD_20260527T082311.SAFE/GRANULE/L2A_T47NPD_A008997_20260527T033930/IMG_DATA/R10m/T47NPD_20260527T032511_B08_10m.jp2


In [14]:
# %%
# Read the Red and NIR satellite bands

import rasterio
from rasterio.env import Env

with Env(AWS_NO_SIGN_REQUEST="YES"):
    with rasterio.open(red_asset.href) as red_src:
        red = red_src.read(1)

    with rasterio.open(nir_asset.href) as nir_src:
        nir = nir_src.read(1)

print("Red shape:", red.shape)
print("NIR shape:", nir.shape)

RasterioIOError: AccessDenied: Access Denied